# Milestone 1: Fire & Hazard Detection with YOLO
### Senior Design Project: Boiler Room Hazard & Component Analysis (CSE21-ME24)

---

## 🎓 What You Will Learn in This Notebook
1. **Computer Vision Basics:** The difference between *Classification* (answering "Is there fire?") and *Object Detection* (answering "Where is the fire located with bounding boxes?").
2. **How YOLO (You Only Look Once) Works:**
   - Splits an image into a grid.
   - Predicts bounding box coordinates $[x, y, w, h]$, confidence score, and class label in a single forward pass.
   - Uses **NMS (Non-Maximum Suppression)** to prune duplicate bounding boxes around the same flame.
3. **Fine-Tuning:** Taking a model pre-trained on millions of general objects (COCO dataset) and adapting its "eyes" specifically to detect industrial fires and smoke.
4. **Evaluation Metrics:** Understanding **Precision**, **Recall**, and **mAP50** (Mean Average Precision).

## Step 1: Check GPU Acceleration
In Google Colab, go to `Runtime` -> `Change runtime type` -> select `T4 GPU`.

In [1]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


CUDA Available: False


## Step 2: Install Ultralytics (YOLO)
`ultralytics` is the official modern library for YOLO models (YOLOv8 & YOLO11).

In [ ]:
!pip install -q ultralytics

## Step 3: Test a Pre-trained YOLO Model
Before training on custom fire data, let's see how YOLO runs out-of-the-box on a test image.

In [2]:
from ultralytics import YOLO
from PIL import Image

# Load lightweight YOLO11 nano model
model = YOLO('yolo11n.pt')

# Run inference on a sample web image
results = model.predict('https://ultralytics.com/images/bus.jpg')
results[0].show()

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\thube\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

image 1/1 c:\Users\thube\SDP\SDP-CSE21-ME24\notebooks\bus.jpg: 640x480 4 persons, 1 bus, 119.8ms
Speed: 8.9ms preprocess, 119.8ms inference, 3.1ms postprocess per image at shape (1, 3, 640, 480)


## Step 4: Download Fire Detection Dataset
To detect fire, we need an annotated dataset. Annotated datasets have:
- `images/`: JPG/PNG pictures of scenes with and without fire.
- `labels/`: Text files containing `class_id x_center y_center width height` (normalized between 0 and 1).
- `data.yaml`: A configuration file telling YOLO where the training and validation images are located.

In [ ]:
# Download a starter fire-and-smoke dataset
# You can also export directly from Roboflow Universe using their python snippet
#!pip install -q roboflow
import os
from roboflow import Roboflow
# TIP: Roboflow has public fire datasets ready in YOLO format.
# Example: Search 'fire detection' on universe.roboflow.com and grab a free API key snippet.
rf = Roboflow(api_key=str(os.getenv("ROBOFLOW_API_KEY")))
project = rf.workspace("mahmoud-9wyf6").project("fire-smoke-detection")
version = project.version(3)
dataset = version.download("yolov11")
                

RuntimeError: API Key is of Incorrect Type 
 Expected Type: <class 'str'>
 Input Type: <class 'NoneType'>

## Step 5: Train / Fine-Tune YOLO on Fire Data
Fine-tuning adjusts the weights of the neural network.
- `epochs`: Number of full passes through the training data (e.g. 20-50).
- `imgsz`: Image resolution fed into the neural network (640x640 is standard).
- `batch`: Number of images processed simultaneously in GPU memory.

In [ ]:
# Replace 'data.yaml' with your dataset yaml path
# model = YOLO('yolo11n.pt')
# results = model.train(data='path/to/data.yaml', epochs=25, imgsz=640, device=0)

## Step 6: Validate and Inspect Metrics
- **Precision:** When the model says "Fire!", how often is it actually fire? (High precision = low false alarms)
- **Recall:** Out of all actual fires in the video, how many did the model catch? (High recall = no missed fires)
- **mAP50:** Mean Average Precision at IoU 0.50 (industry benchmark for detection accuracy).

In [ ]:
# metrics = model.val()
# print(f"mAP50: {metrics.box.map50}")
# print(f"mAP50-95: {metrics.box.map}")

## Step 7: Export Model Weights for Local Use
Once training completes, the best weights are saved at `runs/detect/train/weights/best.pt`.
Download this file and place it in your local `SDP-CSE21-ME24/` project to run live video inference!

In [ ]:
from google.colab import files
# files.download('runs/detect/train/weights/best.pt')